# CS5720 Home Assignment 4

## Part II Programming

Run the setup cell first, then run Q1 through Q3. A CPU runtime is sufficient.

In [1]:
import subprocess
import sys

# Install the NLP libraries and the English spaCy model.
subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "--quiet",
        "numpy", "nltk==3.9.2", "spacy>=3.8,<3.9",
        "https://github.com/explosion/spacy-models/releases/download/"
        "en_core_web_sm-3.8.0/en_core_web_sm-3.8.0-py3-none-any.whl",
    ],
    check=True,
)

import nltk
import numpy as np
import spacy
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import TreebankWordTokenizer

nltk.download("stopwords", quiet=True, raise_on_error=True)
english_stopwords = set(stopwords.words("english"))
tokenizer = TreebankWordTokenizer()
stemmer = PorterStemmer()
nlp = spacy.load("en_core_web_sm")
print(f"Setup complete. spaCy English model {nlp.meta['version']}.")

Setup complete. spaCy English model 3.8.0.


### Q1 NLP Preprocessing Pipeline

The first list includes punctuation. The filtered list keeps word tokens and uses case-insensitive stopword matching. Porter stemming then reduces each remaining word.

In [2]:
def preprocess_sentence(sentence):
    tokens = tokenizer.tokenize(sentence)

    # Keep tokens with letters or digits and remove English stopwords.
    filtered_tokens = [
        token for token in tokens
        if any(character.isalnum() for character in token)
        and token.lower() not in english_stopwords
    ]
    stemmed_words = [stemmer.stem(token) for token in filtered_tokens]

    print("Original Tokens:", tokens)
    print("Tokens Without Stopwords:", filtered_tokens)
    print("Stemmed Words:", stemmed_words)
    return tokens, filtered_tokens, stemmed_words


sentence = "NLP techniques are used in virtual assistants like Alexa and Siri."
original_tokens, filtered_tokens, stems = preprocess_sentence(sentence)

Original Tokens: ['NLP', 'techniques', 'are', 'used', 'in', 'virtual', 'assistants', 'like', 'Alexa', 'and', 'Siri', '.']
Tokens Without Stopwords: ['NLP', 'techniques', 'used', 'virtual', 'assistants', 'like', 'Alexa', 'Siri']
Stemmed Words: ['nlp', 'techniqu', 'use', 'virtual', 'assist', 'like', 'alexa', 'siri']


### Q2 Named Entity Recognition with spaCy

The sentence keeps its leading space so the offsets match the assignment input. Offsets are zero-based, with an exclusive end. Entity labels are predictions from the English model.

In [3]:
def print_named_entities(sentence):
    doc = nlp(sentence)
    for entity in doc.ents:
        print(
            f"{entity.text!r} | {entity.label_} | "
            f"start={entity.start_char} | end={entity.end_char}"
        )
    return doc.ents


sentence = " LeBron James as the NBA player of the United States in Laker and won the NBA Final MVP in 2020."
entities = print_named_entities(sentence)

'LeBron James' | PERSON | start=1 | end=13
'NBA' | ORG | start=21 | end=24
'the United States' | GPE | start=35 | end=52
'Laker' | GPE | start=56 | end=61
'MVP' | ORG | start=84 | end=87
'2020' | DATE | start=91 | end=95


### Q3 Scaled Dot-Product Attention

Each row of the score matrix corresponds to one query. Softmax runs across the keys in each row, and the resulting weights combine the value vectors.

In [4]:
def scaled_dot_product_attention(queries, keys, values):
    queries = np.asarray(queries, dtype=float)
    keys = np.asarray(keys, dtype=float)
    values = np.asarray(values, dtype=float)

    if any(matrix.ndim != 2 for matrix in (queries, keys, values)):
        raise ValueError("Queries, keys, and values must be 2D matrices.")

    key_count, key_dimension = keys.shape
    if key_count == 0 or key_dimension == 0:
        raise ValueError("Keys must have at least one row and one feature.")
    if queries.shape[1] != key_dimension:
        raise ValueError("Queries and keys must have the same feature dimension.")
    if values.shape[0] != key_count:
        raise ValueError("Keys and values must have the same number of rows.")

    scaled_scores = (queries @ keys.T) / np.sqrt(key_dimension)

    # Subtract each row's maximum before exponentiation for stable softmax.
    shifted_scores = scaled_scores - np.max(scaled_scores, axis=1, keepdims=True)
    exponentials = np.exp(shifted_scores)
    weights = exponentials / exponentials.sum(axis=1, keepdims=True)
    return weights, weights @ values


Q = np.array([[1, 0, 1, 0], [0, 1, 0, 1]])
K = np.array([[1, 0, 1, 0], [0, 1, 0, 1]])
V = np.array([[1, 2, 3, 4], [5, 6, 7, 8]])

weights, output = scaled_dot_product_attention(Q, K, V)
with np.printoptions(precision=6, floatmode="fixed"):
    print("Attention weights:")
    print(weights)
    print("Final output:")
    print(output)

Attention weights:
[[0.731059 0.268941]
 [0.268941 0.731059]]
Final output:
[[2.075766 3.075766 4.075766 5.075766]
 [3.924234 4.924234 5.924234 6.924234]]
